# Notebook 03 - Patches, Tokens, Spectrograms, and Budgeting

This notebook turns modality inputs into explicit compute budgets so later serving and evaluation decisions have a measurable foundation.


## What you will learn

- how image resolution affects patch counts
- how audio duration turns into analysis windows
- how frame sampling choices dominate video cost


In [ ]:
!pip install -q numpy pandas matplotlib
print("Installed notebook dependencies.")


In [ ]:
import json
import math
import random
import statistics
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-multimodal")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
NOTEBOOK_ROOT = ARTIFACT_ROOT / "03_patches_tokens_spectrograms_and_budgeting"
NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(7)
np.random.seed(7)



print("Cache dir:", CACHE_DIR)
print("Artifact root:", NOTEBOOK_ROOT.resolve())


## Step 1: Estimate image patch counts

Patch counts are a useful first-order proxy for image cost when the model uses a vision transformer-like encoder.


In [ ]:
def patch_count(width, height, patch=14):
    return math.ceil(width / patch) * math.ceil(height / patch)

image_sizes = pd.DataFrame([
    {"name": "phone screenshot", "width": 1080, "height": 1920},
    {"name": "report page", "width": 1240, "height": 1754},
    {"name": "small crop", "width": 512, "height": 512},
])
image_sizes["patches"] = image_sizes.apply(lambda row: patch_count(row.width, row.height), axis=1)
image_sizes


## Step 2: Translate audio into analysis windows

Speech and sound pipelines usually operate over overlapping windows. The exact frontend differs by model, but the budgeting idea is stable.


In [ ]:
def audio_windows(seconds, window=30, stride=15):
    if seconds <= window:
        return 1
    return 1 + math.ceil((seconds - window) / stride)

audio_clips = pd.DataFrame([
    {"clip": "voice note", "seconds": 18},
    {"clip": "meeting segment", "seconds": 95},
    {"clip": "call snippet", "seconds": 180},
])
audio_clips["analysis_windows"] = audio_clips["seconds"].map(audio_windows)
audio_clips


## Step 3: Budget sampled frames for video

Short-video pipelines stay tractable by sampling frames instead of pretending every frame deserves equal attention.


In [ ]:
def sampled_frames(duration_seconds, fps=1):
    return duration_seconds * fps

video_jobs = pd.DataFrame([
    {"video": "screen recording", "seconds": 45, "fps": 1},
    {"video": "product demo", "seconds": 120, "fps": 1},
    {"video": "inspection clip", "seconds": 90, "fps": 2},
])
video_jobs["sampled_frames"] = video_jobs.apply(lambda row: sampled_frames(row.seconds, row.fps), axis=1)
video_jobs["relative_cost"] = (video_jobs["sampled_frames"] / 32).round(2)
video_jobs


## Exercises and extensions

1. Add a model-specific memory estimate for each row in the video_jobs table.
1. Repeat the patch-count experiment with lower resolutions to quantify the benefit of downscaling or cropping.
